In [6]:
!pip install pyasyncore
!pip install BAC0
!pip install bacpypes
!pip install pytz
!pip install netifaces

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [4]:
import asyncio
import random

import BAC0
from BAC0.core.devices.local.factory import (
    ObjectFactory,
    analog_input,
    analog_value,
    binary_input,
    binary_output,
    multistate_value,
    make_state_text,
)


def add_points(device):
    """
    Create and register simulated BACnet objects on the given device.
    """
    # Start with a clean set of objects
    ObjectFactory.clear_objects()
    objects = []

    # Example Analog Input (Zone Temperature)
    objects.append(
        analog_input(
            name="ZN-T",
            properties={"units": "degreesCelsius"},
            description="Zone Temperature",
            presentValue=21,
        )
    )

    # Example Analog Value (commandable setpoint)
    objects.append(
        analog_value(
            name="Setpoint",
            presentValue=20,
            is_commandable=True,
        )
    )

    # Example Binary Objects (on/off type points)
    objects.append(binary_input(name="DoorStatus", description="Door Open/Closed"))
    objects.append(binary_output(name="FanEnable", description="Fan Control"))

    # Example Multistate Value (Alarm state)
    states = make_state_text(["Normal", "Alarm", "Emergency"])
    objects.append(
        multistate_value(
            description="Alarm State",
            properties={"stateText": states},
            name="BIG-ALARM",
            is_commandable=False,
        )
    )

    # Register all created objects into the BACnet app
    for obj in objects:
        obj.add_objects_to_application(device)


async def main():
    """
    Start BACnet stack and run a simulated BACnet device.
    """
    # Main BACnet network
    async with BAC0.start(localObjName="BACnetMain") as bacnet:
        # Create a simulated device (on port 47809)
        async with BAC0.start(port=47809, localObjName="SimDevice1") as device_app:
            add_points(device_app)

            # Get device IP and ID
            ip = device_app.localIPAddr.addrTuple[0]
            boid = device_app.Boid

            # Connect to the simulated device
            test_device = await BAC0.device(f"{ip}:47809", boid, bacnet, poll=5)

            bacnet._log.info("✅ BACnet simulator started. Press CTRL-C to stop.")

            # Simulation loop
            while True:
                await asyncio.sleep(2)

                # Show current Alarm state
                bacnet._log.info(f"BIG-ALARM = {test_device['BIG-ALARM']}")

                # Randomly update the Setpoint value
                new_val = random.randint(18, 28)
                bacnet._log.info(f"Updating Setpoint to {new_val}")
                await test_device["Setpoint"].write(new_val, priority=16)


if __name__ == "__main__":
    asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop